# PhysDual-GNN — Fase 1: Ingestão de Dados, Splits e Cache ESM-2

Este notebook executa passo a passo o pipeline da **Fase 1** na GPU alocada pelo Colab:
1. Montagem do Google Drive e download direto dos dados do repositório canônico DeepDTA (sem necessidade de credenciais de terceiros) ou via Kagglehub opcional.
2. Validação estrita de procedência (Davis: $68 \times 442 = 30.056$; KIBA: $2.111 \times 229 = 118.254$), padronização e auditoria de valores censurados ($pK_d = 5.0$) e colapso químico/biológico (`by_id` vs. `by_content`).
3. Geração determinística e verificação de não-vazamento por conteúdo dos 30 splits (`random`, `cold_drug`, `cold_target` $\times$ 5 seeds).
4. Featurização molecular em grafos PyG persistidos como dicionários puros de tensores (`weights_only=True`).
5. Extração offline de embeddings do ESM-2 em `fp16` (com janela deslizante `max_len=1022, overlap=256` para proteínas longas).
6. Criação do tarball por modelo (`esm_embeddings_{model_name}.tar`) e execução do verificador de gate (`scripts/verify_phase1.py`).

In [ ]:
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    os.environ.setdefault("PHYSDUAL_ROOT", "/content/drive/MyDrive/physdual-gnn")
    for secret_key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            val = userdata.get(secret_key)
            if val:
                os.environ[secret_key] = str(val).strip()
        except Exception:
            pass
    repo_candidate = Path(os.environ["PHYSDUAL_ROOT"]) / "repo"
    if repo_candidate.exists():
        os.chdir(repo_candidate)

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
for sub in ("src", "scripts"):
    p = str((repo_root / sub).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

from physdual.utils.config import load_config
from physdual.utils.paths import ensure_project_dirs, get_project_root, is_colab

cfg = load_config()
root_dir = get_project_root()
ensure_project_dirs(root_dir)
print(f"Raiz do projeto: {root_dir} | Colab: {is_colab()}")

In [ ]:
import json
from physdual.data.datasets import prepare_dataset

datasets_data = {}
for ds_name in ("davis", "kiba"):
    df, clean_path, report = prepare_dataset(dataset_name=ds_name, affinity_unit="auto", save_clean=True)
    datasets_data[ds_name] = (df, report)
    print(f"=== {ds_name.upper()} ===")
    print(json.dumps({k: v for k, v in report.items() if k != "smiles_failures"}, indent=2))
    display(df.head(3))

In [ ]:
from physdual.data.splits import create_splits, save_splits, verify_split_leakage

splits_dir = Path(cfg["paths"]["processed_data_dir"]) / "splits"
for ds_name, (df, _) in datasets_data.items():
    for st in cfg["splits"]["available"]:
        for seed in cfg["seeds"]:
            sp = create_splits(df, split_type=st, seed=int(seed))
            assert verify_split_leakage(df, sp, split_type=st), f"Vazamento detectado: {ds_name} {st} seed{seed}"
            save_splits(sp, splits_dir / f"{ds_name}_{st}_seed{seed}.json")
print("Todos os 30 splits gerados e validados sem vazamento por conteúdo (SMILES/sequência)!")

In [ ]:
from physdual.data.featurizer_drug import build_drug_graph_cache

processed_dir = Path(cfg["paths"]["processed_data_dir"])
for ds_name, (df, _) in datasets_data.items():
    uniq = df[["drug_id", "smiles"]].drop_duplicates(subset=["smiles"])
    g_path, g_rep = build_drug_graph_cache(
        smiles_list=uniq["smiles"].tolist(),
        output_path=processed_dir / f"{ds_name}_drug_graphs.pt",
        drug_ids=uniq["drug_id"].tolist(),
    )
    print(f"[{ds_name}] Cache de grafos salvo em {g_path}: {g_rep}")

In [ ]:
import run_phase1_data

# Executa o orquestrador completo (reutilizando o que já estiver salvo e gerando o relatório final)
rc = run_phase1_data.main(["--dataset", "all"])
assert rc == 0, f"run_phase1_data falhou com código {rc}"

In [ ]:
import verify_phase1

gate_rc = verify_phase1.main([])
assert gate_rc == 0, "O Gate de Pronto da Fase 1 reprovou um ou mais itens!"